# Favorita TFT evaluation on Google Colab GPU

This notebook trains a compact **Temporal Fusion Transformer (TFT)** with PyTorch Forecasting on the existing Favorita sample. It preserves the active model set and scores the exact **116 item-store series × 28 validation dates = 3,248 observations** used by the primary Naive Weekly and LightGBM validation.

- **Target:** daily `unit_sales`.
- **Training ends:** 2017-07-18 (training target dates are strictly before 2017-07-19).
- **Validation:** 2017-07-19 through 2017-08-15.
- **Evaluation forecast horizon:** one day ahead, rolling. Each date is predicted using the observed target history through the previous date, matching the active pipeline’s validation feature protocol. The validation window is 28 days.
- **Known inputs:** calendar and promotion features plus the active pipeline’s lagged/rolling sales and lag-1 oil/transactions values. Same-day sales are not model inputs. Static item/store metadata are categorical.
- **Validation target values:** retained in the uploaded export only for scoring. Training data and target normalization scales stop before validation.

## Before running

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. Upload this notebook and run cells in order.
3. When prompted, upload `tft_favorita_116_series.csv` from `data/processed/` in the project.
4. The final cell downloads a ZIP with actual predictions, metrics, checkpoint, training history, and execution report.

The notebook does not change the local dashboard. It writes a success report and metrics only after training and held-out prediction finish; failures are written as failed execution reports without invented scores.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise FileNotFoundError("Upload tft_favorita_116_series.csv")
DATA_PATH = csv_names[0]
print("Uploaded dataset:", DATA_PATH)

In [ ]:
%pip install -q "pytorch-forecasting>=1.4,<2" "lightning>=2,<3" "pandas>=2,<3.1" "scikit-learn>=1.3,<2"

In [ ]:
import json
import math
import os
import random
import shutil
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

OUTPUT_DIR = Path("/content/tft_colab_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "tft_execution_report.json"
REPORT = {
    "status": "running",
    "stage": "initialization",
    "model": "TemporalFusionTransformer",
    "target": "unit_sales",
    "seed": SEED,
    "device": "CUDA" if torch.cuda.is_available() else "CPU",
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

def write_report(status=None, stage=None, error=None, **fields):
    if status is not None:
        REPORT["status"] = status
    if stage is not None:
        REPORT["stage"] = stage
    if error is not None:
        REPORT["error"] = error
    REPORT.update(fields)
    REPORT_PATH.write_text(json.dumps(REPORT, indent=2, default=str))

write_report()
print("PyTorch:", torch.__version__)
print("PyTorch Forecasting:", __import__("pytorch_forecasting").__version__)
print("Lightning:", pl.__version__)
print("CUDA:", torch.cuda.is_available(), REPORT["gpu_name"])

## Load and verify the exported data

The CSV contains the exact current series and real panel history. `rmsse_scale` is a training-only metric denominator matching `src/evaluation/metrics.py`; it is never passed as a model feature. The existing features `lag_1`, `lag_7`, `lag_14`, `lag_28`, `mean_7`, `mean_28`, and `std_28` are shifted/rolling prior-sales features. Oil and transactions are lagged by one day. Validation rows are marked as unknown target values to TFT, although their labels remain available separately for metric calculation.

In [ ]:
try:
    REPORT["stage"] = "data_validation"
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    required = {
        "series_id", "id", "date", "time_idx", "unit_sales", "rmsse_scale",
        "lag_1", "lag_7", "lag_14", "lag_28", "mean_7", "mean_28", "std_28",
        "onpromotion", "is_holiday", "oil_price_lag_1", "transactions_lag_1",
        "dow_cat", "month_cat", "day_cat", "weekofyear_cat",
        "item_cat", "store_cat", "family_cat", "class_cat", "perishable_cat",
        "state_cat", "cluster_cat", "type_cat", "store_type_cat",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Export is missing required columns: {missing}")
    df["series_id"] = df.series_id.astype(str)
    df["id"] = df.id.astype(str)
    df = df.sort_values(["series_id", "time_idx"]).reset_index(drop=True)

    VALIDATION_START = pd.Timestamp("2017-07-19")
    VALIDATION_END = pd.Timestamp("2017-08-15")
    TRAIN_END = VALIDATION_START - pd.Timedelta(days=1)
    EXPECTED_SERIES = 116
    EXPECTED_VALIDATION_ROWS = 3248
    validation_actuals = df[df.date.between(VALIDATION_START, VALIDATION_END)].copy()
    series_ids = sorted(df.series_id.unique().tolist())
    if len(series_ids) != EXPECTED_SERIES:
        raise ValueError(f"Expected {EXPECTED_SERIES} series; found {len(series_ids)}")
    if len(validation_actuals) != EXPECTED_VALIDATION_ROWS:
        raise ValueError(f"Expected {EXPECTED_VALIDATION_ROWS} held-out rows; found {len(validation_actuals)}")
    if validation_actuals.series_id.nunique() != EXPECTED_SERIES:
        raise ValueError("Validation does not include the exact expected series count")
    counts = validation_actuals.groupby("series_id").size()
    if not counts.eq(28).all():
        raise ValueError("Every series must have all 28 validation dates")
    if df.duplicated(["series_id", "date"]).any():
        raise ValueError("Duplicate series-date rows found")
    if df["rmsse_scale"].isna().any() or (df.rmsse_scale <= 0).any():
        raise ValueError("Invalid training-only RMSSE scale")
    known_real = ["lag_1", "lag_7", "lag_14", "lag_28", "mean_7", "mean_28", "std_28",
                  "onpromotion", "is_holiday", "oil_price_lag_1", "transactions_lag_1"]
    model_df = df.dropna(subset=known_real).copy()
    if model_df[known_real].isna().any().any():
        raise ValueError("Missing values remain after excluding the lag-feature warm-up period")
    # Check the earliest evaluation input: lag_1 equals the preceding day actual.
    check = validation_actuals[["id", "date", "lag_1"]].merge(
        df[["id", "date", "unit_sales"]].assign(date=lambda x: x.date + pd.Timedelta(days=1)),
        on=["id", "date"], how="left", validate="one_to_one")
    if check.unit_sales.isna().any() or not np.array_equal(check.lag_1.to_numpy(), check.unit_sales.to_numpy()):
        raise ValueError("Lag-1 contains same-day/future target information")

    TRAIN_CUTOFF = VALIDATION_START
    training_rows = model_df[model_df.date < TRAIN_CUTOFF].copy()
    if training_rows.date.max() != TRAIN_END:
        raise ValueError(f"Training must end on {TRAIN_END.date()}")
    val_start_idx = int(model_df.loc[model_df.date.eq(VALIDATION_START), "time_idx"].min())
    time_to_date = model_df.drop_duplicates("time_idx").set_index("time_idx").date.to_dict()
    REPORT.update({
        "dataset_file": str(DATA_PATH), "series_count": EXPECTED_SERIES,
        "dataset_rows": int(len(df)), "model_rows_after_feature_warmup": int(len(model_df)),
        "feature_warmup_rows_excluded_from_windows": int(len(df) - len(model_df)),
        "training_rows": int(len(training_rows)),
        "training_start": str(training_rows.date.min().date()),
        "training_end": str(training_rows.date.max().date()),
        "validation_start": str(VALIDATION_START.date()), "validation_end": str(VALIDATION_END.date()),
        "validation_window_days": 28, "forecast_horizon_days": 1,
        "validation_observations": int(len(validation_actuals)),
        "series_ids": series_ids,
        "leakage_checks": {"lag_1_is_prior_day": True, "same_day_target_excluded_from_known_inputs": True},
    })
    write_report(stage="data_validation_complete")
    print(f"Verified {len(series_ids)} series, {len(validation_actuals)} held-out observations.")
    print(f"Train through {TRAIN_END.date()}; rolling one-step validation {VALIDATION_START.date()} to {VALIDATION_END.date()}.")
except Exception as exc:
    write_report(status="failed", stage="data_validation", error=f"{type(exc).__name__}: {exc}")
    raise

## Build one-day rolling `TimeSeriesDataSet`

The validation dataset creates one decoder step per validation date. For each window, TFT receives target history only through the preceding day; validation outcomes are never registered as known covariates. This matches the active pipeline's daily rolling one-step validation and produces the same 3,248 series-date keys.

In [ ]:
try:
    REPORT["stage"] = "dataset_construction"
    STATIC_CATEGORICALS = ["item_cat", "store_cat", "family_cat", "class_cat", "perishable_cat",
                           "state_cat", "cluster_cat", "type_cat", "store_type_cat"]
    KNOWN_CATEGORICALS = ["dow_cat", "month_cat", "day_cat", "weekofyear_cat"]
    KNOWN_REALS = ["time_idx", "lag_1", "lag_7", "lag_14", "lag_28", "mean_7", "mean_28", "std_28",
                   "onpromotion", "is_holiday", "oil_price_lag_1", "transactions_lag_1"]
    CATEGORICALS = STATIC_CATEGORICALS + KNOWN_CATEGORICALS
    for col in CATEGORICALS:
        df[col] = df[col].astype(str)
    model_df = df.dropna(subset=KNOWN_REALS).copy()
    training_rows = model_df[model_df.date < VALIDATION_START].copy()

    train_dataset = TimeSeriesDataSet(
        training_rows,
        time_idx="time_idx",
        target="unit_sales",
        group_ids=["series_id"],
        min_encoder_length=56,
        max_encoder_length=56,
        min_prediction_length=1,
        max_prediction_length=1,
        static_categoricals=STATIC_CATEGORICALS,
        time_varying_known_categoricals=KNOWN_CATEGORICALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_unknown_reals=["unit_sales"],
        target_normalizer=GroupNormalizer(groups=["series_id"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=False,
        randomize_length=False,
    )
    validation_dataset = TimeSeriesDataSet.from_dataset(
        train_dataset,
        model_df,
        min_prediction_idx=val_start_idx,
        predict=False,
        stop_randomization=True,
    )
    if len(validation_dataset) != EXPECTED_VALIDATION_ROWS:
        raise ValueError(f"Expected {EXPECTED_VALIDATION_ROWS} one-step validation windows; found {len(validation_dataset)}")
    BATCH_SIZE = 32
    train_loader = train_dataset.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
    validation_loader = validation_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
    REPORT.update({"model_rows_after_feature_warmup": int(len(model_df)),
                   "train_windows": int(len(train_dataset)), "validation_windows": int(len(validation_dataset)),
                   "batch_size": BATCH_SIZE, "encoder_length_days": 56,
                   "known_real_covariates": KNOWN_REALS,
                   "known_categorical_covariates": KNOWN_CATEGORICALS,
                   "static_categorical_covariates": STATIC_CATEGORICALS})
    write_report(stage="dataset_construction_complete")
    print("Training windows:", len(train_dataset), "Validation windows:", len(validation_dataset))
except Exception as exc:
    write_report(status="failed", stage="dataset_construction", error=f"{type(exc).__name__}: {exc}")
    raise

## Train compact TFT on CUDA

The requested T4-friendly configuration is hidden size 8, two attention heads, batch size 32, maximum 15 epochs, and early stopping on validation loss (patience 3). QuantileLoss trains probabilistic outputs; `mode="prediction"` yields the point prediction used for WAPE/RMSSE/RMSE.

In [ ]:
try:
    REPORT["stage"] = "training"
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is unavailable. In Colab, select Runtime > Change runtime type > T4 GPU and rerun.")
    device_name = torch.cuda.get_device_name(0)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    csv_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name="lightning_logs")
    checkpoint_callback = ModelCheckpoint(
        dirpath=str(OUTPUT_DIR), filename="tft-best-epoch{epoch:02d}",
        monitor="val_loss", mode="min", save_top_k=1, save_last=False,
    )
    early_stopping = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=3, mode="min", verbose=True)
    model = TemporalFusionTransformer.from_dataset(
        train_dataset,
        learning_rate=3e-3,
        hidden_size=8,
        attention_head_size=2,
        dropout=0.1,
        hidden_continuous_size=8,
        loss=QuantileLoss(),
        log_interval=100,
        reduce_on_plateau_patience=2,
    )
    trainer = pl.Trainer(
        max_epochs=15,
        accelerator="gpu",
        devices=1,
        callbacks=[early_stopping, checkpoint_callback],
        logger=csv_logger,
        gradient_clip_val=0.1,
        enable_model_summary=False,
        log_every_n_steps=100,
        deterministic=True,
    )
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=validation_loader)
    checkpoint_path = checkpoint_callback.best_model_path
    if not checkpoint_path or not Path(checkpoint_path).is_file():
        raise RuntimeError("Training ended without a best-model checkpoint")
    shutil.copy2(checkpoint_path, OUTPUT_DIR / "tft_model.ckpt")
    history_path = Path(csv_logger.log_dir) / "metrics.csv"
    if not history_path.is_file():
        raise RuntimeError("Lightning training history was not created")
    shutil.copy2(history_path, OUTPUT_DIR / "tft_training_history.csv")
    best_model = TemporalFusionTransformer.load_from_checkpoint(str(OUTPUT_DIR / "tft_model.ckpt"))
    REPORT.update({
        "device": "CUDA", "gpu_name": device_name,
        "architecture": {"hidden_size": 8, "attention_head_size": 2},
        "max_epochs": 15, "epochs_completed": int(trainer.current_epoch + 1),
        "early_stopping_patience": 3, "best_checkpoint": str(OUTPUT_DIR / "tft_model.ckpt"),
        "best_checkpoint_original": checkpoint_path,
        "best_validation_loss": float(checkpoint_callback.best_model_score.detach().cpu()),
        "training_history": str(OUTPUT_DIR / "tft_training_history.csv"),
    })
    write_report(stage="training_complete")
    print("GPU:", device_name, "Epochs completed:", trainer.current_epoch + 1)
    print("Best checkpoint:", OUTPUT_DIR / "tft_model.ckpt")
except Exception as exc:
    write_report(status="failed", stage="training", error=f"{type(exc).__name__}: {exc}")
    raise

## Held-out predictions and metrics

Predictions are joined back to the exported actuals by series and date. The notebook fails rather than score if any held-out key is missing/duplicated or if the keys differ from the active validation CSV specification. RMSSE uses the exported training-only scale, computed with the same formula as the existing metric implementation.

In [ ]:
try:
    REPORT["stage"] = "heldout_prediction_and_scoring"
    raw_predictions, prediction_index = best_model.predict(
        validation_loader,
        mode="prediction",
        return_index=True,
        trainer_kwargs={"accelerator": "gpu", "devices": 1, "logger": False, "enable_progress_bar": False},
    )
    pred_array = raw_predictions.detach().cpu().numpy() if torch.is_tensor(raw_predictions) else np.asarray(raw_predictions)
    pred_array = pred_array.reshape(len(prediction_index), -1)
    if pred_array.shape[1] != 1:
        raise ValueError(f"Expected one prediction per rolling window; output shape was {pred_array.shape}")
    prediction_index = prediction_index.copy()
    if "series_id" not in prediction_index or "time_idx" not in prediction_index:
        raise ValueError(f"Unexpected TFT prediction index columns: {prediction_index.columns.tolist()}")
    prediction_index["series_id"] = prediction_index.series_id.astype(str)
    prediction_index["time_idx"] = prediction_index.time_idx.astype(int)
    prediction_index["prediction"] = np.maximum(pred_array[:, 0], 0.0)
    prediction_index["date"] = prediction_index.time_idx.map(time_to_date)
    scored = prediction_index[["series_id", "date", "prediction"]].merge(
        df[["series_id", "id", "date", "unit_sales", "store_nbr", "item_nbr", "rmsse_scale"]],
        on=["series_id", "date"], how="left", validate="one_to_one",
    )
    scored = scored.rename(columns={"unit_sales": "actual"})
    if scored.actual.isna().any() or len(scored) != EXPECTED_VALIDATION_ROWS:
        raise ValueError(f"TFT returned {len(scored)} rows with {int(scored.actual.isna().sum())} missing actuals")
    scored = scored[scored.date.between(VALIDATION_START, VALIDATION_END)].copy()
    expected_keys = validation_actuals[["series_id", "date"]].sort_values(["series_id", "date"]).reset_index(drop=True)
    actual_keys = scored[["series_id", "date"]].sort_values(["series_id", "date"]).reset_index(drop=True)
    if not expected_keys.equals(actual_keys):
        raise ValueError("TFT series-date keys do not match the existing validation observations")

    errors = scored.actual.to_numpy(dtype=float) - scored.prediction.to_numpy(dtype=float)
    denominator = float(np.abs(scored.actual.to_numpy(dtype=float)).sum())
    wape = float(np.abs(errors).sum() / denominator) if denominator else float("nan")
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    per_series = []
    for sid, group in scored.groupby("series_id", sort=False):
        scale = float(group.rmsse_scale.iloc[0])
        if scale > 0:
            per_series.append(float(np.sqrt(np.mean((group.actual.to_numpy() - group.prediction.to_numpy()) ** 2) / scale)))
    rmsse_mean = float(np.mean(per_series)) if per_series else float("nan")
    scored["residual"] = scored.actual - scored.prediction
    prediction_path = OUTPUT_DIR / "tft_predictions.csv"
    scored[["date", "id", "store_nbr", "item_nbr", "actual", "prediction", "residual"]].to_csv(prediction_path, index=False)
    metrics = {
        "model": "Temporal Fusion Transformer",
        "target": "unit_sales",
        "forecast_horizon_days": 1,
        "evaluation_protocol": "rolling one-day-ahead, actual history through prior day",
        "training_end": str(TRAIN_END.date()),
        "validation_start": str(VALIDATION_START.date()),
        "validation_end": str(VALIDATION_END.date()),
        "validation_window_days": 28,
        "series_count": int(scored.series_id.nunique()),
        "observations": int(len(scored)),
        "wape": wape,
        "rmsse_mean": rmsse_mean,
        "rmse": rmse,
        "metric_definition": "src/evaluation/metrics.py WAPE/RMSE/RMSSE formulas; RMSSE mean of per-series scores using pre-validation training scale",
        "predictions_csv": str(prediction_path),
        "checkpoint": str(OUTPUT_DIR / "tft_model.ckpt"),
    }
    metrics_path = OUTPUT_DIR / "tft_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))
    write_report(status="success", stage="complete", metrics_file=str(metrics_path),
                 predictions_file=str(prediction_path), observations=int(len(scored)),
                 series_evaluated=int(scored.series_id.nunique()))
    print(json.dumps(metrics, indent=2))
except Exception as exc:
    write_report(status="failed", stage="heldout_prediction_and_scoring", error=f"{type(exc).__name__}: {exc}")
    raise

## Download artifacts

A successful run produces predictions, metrics, the best checkpoint, the CSVLogger training history, and a JSON execution report. The ZIP is created only after all outputs have been written.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

required_outputs = [
    OUTPUT_DIR / "tft_predictions.csv",
    OUTPUT_DIR / "tft_metrics.json",
    OUTPUT_DIR / "tft_model.ckpt",
    OUTPUT_DIR / "tft_training_history.csv",
    OUTPUT_DIR / "tft_execution_report.json",
]
missing_outputs = [str(path) for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(f"Not all required outputs exist: {missing_outputs}")
zip_path = OUTPUT_DIR / "tft_colab_artifacts.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in required_outputs:
        archive.write(path, arcname=path.name)
print("Created:", zip_path)
files.download(str(zip_path))

## Interpretation and limitations

The checkpoint and metrics are only valid if `tft_execution_report.json` has `"status": "success"`. If training or scoring fails, inspect its error and do not report a TFT score. This validates a rolling one-step model over the same 3,248 observations; it does not estimate recursive 16-day official-test performance. Cost assumptions and inventory optimization are outside this notebook.